# Weekly project 6
Today we will continue work from monday.
We will follow the style of last week.

Weekly project:
- You will need to implement your own k-means algorithm. You are not allowed to use the implementation in `sklearn`.
- It should be able to cluster each of the different figures.
- Extend your k-means so it finds the optimal amount of clusters.
  
## Challenge
- Implement the mean shift clustering algorithm


In [60]:
import numpy as np
import open3d as o3d
import copy
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import http.server
import socketserver
import tempfile
import threading
from functools import partial
from pathlib import Path

# Browser-based viewer: serves an interactive Plotly page on localhost.
_point_cloud_servers = []

def serve_point_cloud(pcd, title="Point Cloud", point_size=2):
    points = np.asarray(pcd.points)
    if len(points) == 0:
        raise ValueError("Cannot visualize an empty point cloud")
    colors = np.asarray(pcd.colors)
    marker_color = (
        [f"rgb({r},{g},{b})" for r, g, b in np.clip(colors * 255, 0, 255).astype(int)]
        if len(colors) == len(points) else "#1f77b4"
    )
    fig = go.Figure(go.Scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2], mode="markers",
        marker=dict(size=point_size, color=marker_color),
    ))
    fig.update_layout(title=title, scene_aspectmode="data")
    directory = Path(tempfile.mkdtemp(prefix="pointcloud_viewer_"))
    fig.write_html(directory / "index.html", include_plotlyjs=True, full_html=True)
    server = socketserver.TCPServer(("127.0.0.1", 0), partial(http.server.SimpleHTTPRequestHandler, directory=directory))
    threading.Thread(target=server.serve_forever, daemon=True).start()
    _point_cloud_servers.append(server)
    url = f"http://127.0.0.1:{server.server_address[1]}/index.html"
    print(f"Viewer running at: {url}")
    return url

def draw_labels_on_model(pcl, labels):
    cmap = plt.get_cmap("tab20")
    pcl_temp = copy.deepcopy(pcl)
    max_label = labels.max()
    colors = cmap(labels / (max_label if max_label > 0 else 1))
    colors[labels < 0] = 0
    pcl_temp.colors = o3d.utility.Vector3dVector(colors[:, :3])
    return serve_point_cloud(pcl_temp, title="Clustered point cloud")

d = 4
mesh = o3d.geometry.TriangleMesh.create_tetrahedron().translate((-d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_octahedron().translate((0, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_icosahedron().translate((d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_torus().translate((-d, -d, 0))
mesh += o3d.geometry.TriangleMesh.create_mobius(twists=1).translate((0, -d, 0))
mesh += o3d.geometry.TriangleMesh.create_mobius(twists=2).translate((d, -d, 0))

def find_optimal_k(mesh, min_triangles=1):
    """Return the number of disconnected figures in this synthetic mesh."""
    # Each input figure was created as a separate, unconnected triangle mesh.
    _, triangle_counts, _ = mesh.cluster_connected_triangles()
    k = int(np.count_nonzero(np.asarray(triangle_counts) >= min_triangles))
    if k < 2:
        raise ValueError("Need at least two disconnected mesh components to infer k")
    return k

## apply k means on this
point_cloud = mesh.sample_points_uniformly(int(1e3))
optimal_k = find_optimal_k(mesh)
print(f"Detected {optimal_k} disconnected figures; using k={optimal_k}.")
#serve_point_cloud(point_cloud, title="Combined shapes")

Detected 6 disconnected figures; using k=6.


## Choosing the number of clusters

K-means requires the number of clusters, $k$, before it starts. In this synthetic scene, each geometric figure is a separate connected component of the input triangle mesh. `find_optimal_k` therefore uses Open3D's `cluster_connected_triangles()` to count the disconnected components before the mesh is sampled into points.

This detects the six intended figures automatically, so `optimal_k` becomes 6 without hard-coding that number. This approach is appropriate here because the original mesh is available and its connected components represent the objects we want to identify. For an arbitrary unconnected point cloud without mesh topology, a different model-selection criterion would be needed.


In [61]:
def kmeans(k, point_cloud, n_init=20, max_iter=300, tol=1e-5, random_state=0):
    """Cluster an Open3D point cloud with k-means++ and multiple restarts."""
    points = np.asarray(point_cloud.points, dtype=float)
    if not 1 <= k <= len(points):
        raise ValueError("k must be between 1 and the number of points")

    def initialise_centers(rng):
        # k-means++ spreads initial centres across the point cloud.
        centers = np.empty((k, points.shape[1]))
        centers[0] = points[rng.integers(len(points))]
        closest_sq_distances = np.sum((points - centers[0]) ** 2, axis=1)
        for cluster in range(1, k):
            probabilities = closest_sq_distances / closest_sq_distances.sum()
            centers[cluster] = points[rng.choice(len(points), p=probabilities)]
            candidate_sq_distances = np.sum((points - centers[cluster]) ** 2, axis=1)
            closest_sq_distances = np.minimum(closest_sq_distances, candidate_sq_distances)
        return centers

    master_rng = np.random.default_rng(random_state)
    best_labels, best_inertia = None, np.inf

    for _ in range(n_init):
        rng = np.random.default_rng(master_rng.integers(2**32))
        centers = initialise_centers(rng)

        # Alternate between assigning points and updating their centres.
        for _ in range(max_iter):
            sq_distances = np.sum((points[:, np.newaxis] - centers) ** 2, axis=2)
            labels = np.argmin(sq_distances, axis=1)
            new_centers = centers.copy()
            for cluster in range(k):
                members = points[labels == cluster]
                # Re-seed an empty cluster at the currently worst-fit point.
                new_centers[cluster] = (
                    members.mean(axis=0) if len(members) else points[sq_distances.min(axis=1).argmax()]
                )
            if np.max(np.linalg.norm(new_centers - centers, axis=1)) <= tol:
                centers = new_centers
                break
            centers = new_centers

        sq_distances = np.sum((points[:, np.newaxis] - centers) ** 2, axis=2)
        labels = np.argmin(sq_distances, axis=1)
        # Inertia is the total squared distance to each point's assigned centre.
        inertia = sq_distances[np.arange(len(points)), labels].sum()
        if inertia < best_inertia:
            best_labels, best_inertia = labels, inertia

    return best_labels

## Custom K-means: stable initialization and repeated runs

Each k-means iteration has two steps: (1) assign every point to its nearest centre and (2) move each centre to the mean of its assigned points. The process stops when the largest centre movement is at most `tol`.

### Why k-means++?

A single random starting arrangement can put multiple centres on one large figure and leave another figure without a centre. K-means++ chooses the first centre randomly, then favours points far from the existing centres for later centres. This gives the algorithm a much more spatially spread-out starting guess.

K-means can still converge to a local optimum, even with k-means++. Therefore the algorithm is run 20 times with different seeded initial centres. For each run it calculates **inertia**, the sum of squared distances from every point to its assigned centre, and returns the labels from the run with the lowest inertia. The fixed `random_state=0` makes the hand-in reproducible.

If a cluster becomes empty, the implementation re-seeds its centre at the currently worst-represented point. This prevents an empty cluster from producing an invalid mean (`NaN`).


In [62]:
k = optimal_k
labels = kmeans(k, point_cloud, n_init=20, random_state=0)
draw_labels_on_model(point_cloud, labels)


Viewer running at: http://127.0.0.1:39733/index.html


'http://127.0.0.1:39733/index.html'

## Challenge: Mean shift clustering

Mean shift is a density-based clustering algorithm. Rather than choosing $k$ first, it starts from a candidate centre and repeatedly moves it to the mean of nearby points. When the centre stops moving, it has reached a local high-density region, called a **mode**. Points that converge to the same mode receive the same cluster label.

The important parameter is `bandwidth`: the radius of the local neighbourhood. A small bandwidth preserves many small surface-density modes; a large bandwidth merges nearby modes. For these figures, `bandwidth=2.0` keeps the separated objects distinct while merging modes that belong to the same figure. Unlike k-means, the number of output clusters is determined by the density structure and bandwidth, not supplied in advance.


In [ ]:
def mean_shift(point_cloud, bandwidth=2.0, max_iter=100, tol=1e-3, merge_radius=None):
    """Cluster a point cloud by shifting every seed towards a density mode."""
    points = np.asarray(point_cloud.points, dtype=float)
    if bandwidth <= 0:
        raise ValueError("bandwidth must be positive")
    if merge_radius is None:
        merge_radius = bandwidth / 2

    shifted_points = np.empty_like(points)
    densities = np.empty(len(points), dtype=int)

    for index, seed in enumerate(points):
        centre = seed.copy()
        for _ in range(max_iter):
            squared_distances = np.sum((points - centre) ** 2, axis=1)
            neighbours = points[squared_distances <= bandwidth ** 2]
            new_centre = neighbours.mean(axis=0)
            # A mode has been reached when the centre changes by less than tol.
            if np.linalg.norm(new_centre - centre) <= tol:
                centre = new_centre
                break
            centre = new_centre
        shifted_points[index] = centre
        densities[index] = len(neighbours)

    # Merge duplicate modes, considering dense modes first.
    mode_centres = []
    for index in np.argsort(densities)[::-1]:
        if not mode_centres or np.min(np.linalg.norm(np.asarray(mode_centres) - shifted_points[index], axis=1)) > merge_radius:
            mode_centres.append(shifted_points[index])

    mode_centres = np.asarray(mode_centres)
    # Label by the converged (shifted) position, not the original point.
    # This preserves the density mode that each point actually reached.
    squared_distances = np.sum((shifted_points[:, np.newaxis] - mode_centres) ** 2, axis=2)
    labels = np.argmin(squared_distances, axis=1)
    return labels, mode_centres

mean_shift_labels, mean_shift_centres = mean_shift(point_cloud, bandwidth=2.0)
print(f"Mean shift found {len(mean_shift_centres)} clusters.")
draw_labels_on_model(point_cloud, mean_shift_labels)

Mean shift found 6 clusters.
Viewer running at: http://127.0.0.1:41299/index.html


'http://127.0.0.1:41299/index.html'

127.0.0.1 - - [08/Oct/2026 14:33:48] "GET /index.html HTTP/1.1" 200 -
127.0.0.1 - - [08/Oct/2026 14:33:49] code 404, message File not found
127.0.0.1 - - [08/Oct/2026 14:33:49] "GET /favicon.ico HTTP/1.1" 404 -


In [64]:
# Stop and release every localhost viewer started by serve_point_cloud.
for server in _point_cloud_servers:
    server.shutdown()
    server.server_close()
_point_cloud_servers.clear()
print("All local point-cloud viewers stopped.")

All local point-cloud viewers stopped.
